# Stage 04 · Temporal-holdout pre-departure forecast

Forecast memakai service call dermaga yang lolos boundary, continuity, duration, dan interpolation-quality gate dari Stage 03. Episode tersensor, short contact, dan extended stay tetap tersedia sebagai audit, namun tidak dipakai sebagai riwayat perjalanan atau target kedatangan.

In [ ]:
# Shared paths and deterministic stage initialization
from pathlib import Path
from datetime import datetime, timezone
import os, sys
import pandas as pd

root = Path(os.environ.get("MFAR_CODE_ROOT", Path.cwd()))
if not (root / "src" / "mfar_paths.py").is_file():
    root = Path.cwd().parent
if not (root / "src" / "mfar_paths.py").is_file():
    raise FileNotFoundError("Run the notebook from the repository or set MFAR_CODE_ROOT")
sys.path.insert(0, str(root.resolve()))

from src.mfar_paths import *
from src.mfar_core import *
STARTED_AT = datetime.now(timezone.utc)


In [ ]:
NOTEBOOK_NAME = "04_No_Intervention_Forecast.ipynb"
validate_writable_directory(STAGE_DIRS[4], NOTEBOOK_NAME, 4)


In [ ]:
state = validate_csv_input(STAGE_03_DIR / "03_input_state_enhanced.csv",
    ["grid_time", "mmsi", "operational_status", "origin", "destination", "is_at_berth",
     "predicted_berth_release_time", "operational_phase"], NOTEBOOK_NAME, 4)
for col in ["grid_time", "predicted_berth_release_time"]:
    state[col] = pd.to_datetime(state[col], errors="coerce")
episodes = validate_csv_input(STAGE_03_DIR / "03_service_call_history.csv",
    ["mmsi", "berth_entry_time", "observed_end", "port_id",
     "episode_class", "eligible_for_turnaround_calibration"], NOTEBOOK_NAME, 4)
rates = validate_csv_input(VEHICLE_ARRIVAL_PATH,
    ["port_id", "time_start", "time_end", "car_arrival_rate_30min", "motorcycle_arrival_rate_30min"], NOTEBOOK_NAME, 4)
profiles = pd.read_csv(CONFIG_DIR / "vessel_profiles.csv")
berths = pd.read_csv(CONFIG_DIR / "terminal_berths.csv")
queue, event_log, forecast, summary, eta_audit, departure_audit = run_stage4(
    state, episodes, rates, profiles, berths, STAGE_04_DIR, CONFIG_DIR)
display(summary); display(eta_audit); display(departure_audit)


In [ ]:
NOTEBOOK_NAME = "04_No_Intervention_Forecast.ipynb"
stage_dir = STAGE_DIRS[4]
files = sorted(stage_dir.glob("04_*"))
write_execution_metadata(4, NOTEBOOK_NAME, STARTED_AT,
    [CONFIG_DIR / "pipeline_parameters.csv", STAGE_03_DIR / "03_service_call_history.csv"], {},
    {"primary": len(locals().get("state", locals().get("forecast", locals().get("sim", []))))}, files)
